# Analisis Waktu Pengiriman & Efisiensi Kurir
### Take Home Test — Data Analyst | Dataset: Zomato Food Delivery

Notebook ini merangkum seluruh proses analisis dari awal sampai akhir, mengikuti 6 tahap:
1. Problem Statement & Tujuan Analisis
2. Data Preprocessing
3. Exploratory Data Analysis (EDA)
4. Insight & Interpretasi
5. Dashboard (disiapkan sebagai file CSV untuk Power BI/Tableau)
6. Rekomendasi Insight Bisnis

**Catatan:** pastikan file `Zomato_Dataset.csv` berada di folder yang sama dengan notebook ini sebelum menjalankan.


## 0. Setup & Import Library

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

DATA_PATH = "Zomato_Dataset.csv"


## 1. Problem Statement & Tujuan Analisis

**Latar Belakang:** Perusahaan adalah platform *on-demand food delivery marketplace* yang beroperasi
lintas 20+ kota di India, menghubungkan restoran, pelanggan, dan kurir mitra (gig worker) lewat aplikasi.

**Problem Statement:** Waktu pengiriman bervariasi signifikan dan belum diketahui secara pasti faktor
operasional mana yang paling memengaruhinya — sehingga perusahaan kesulitan menentukan kebijakan alokasi
kurir dan kesiapan operasional yang tepat, terutama pada kondisi permintaan tinggi seperti festival, tanpa
mengorbankan kualitas layanan.

**Tujuan Analisis:**
1. Mengidentifikasi & mengurutkan faktor yang paling memengaruhi waktu pengiriman — memisahkan faktor
   yang bisa dikendalikan perusahaan dari faktor eksternal yang harus diantisipasi.
2. Menemukan titik trade-off strategi *order batching* (`multiple_deliveries`).
3. Mengukur dampak periode permintaan tinggi (festival) terhadap waktu pengiriman.
4. Menyusun rekomendasi operasional berbasis data.

**Ruang Lingkup & Batasan:**
- Bersifat operasional/logistik — tidak mencakup dampak finansial (tidak ada kolom harga di dataset).
- Temuan bersifat korelasional, bukan sebab-akibat murni.
- `Delivery_person_ID` tidak dipakai untuk klaim performa kurir individual (indikasi ID adalah kombinasi
  kode yang di-generate, bukan identitas persisten — dibuktikan pada bagian Data Understanding).
- Dataset terindikasi sintetis/edukasi — insight diposisikan sebagai simulasi kasus bisnis.


## 2. Data Understanding

### 2.1 Muat Data & Gambaran Umum

In [2]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


Shape: (45584, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,13-02-2022,09:20,09:30,Sandstorms,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,14-02-2022,19:50,20:05,Fog,Jam,1,Snack,scooter,1.0,No,Metropolitian,41


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 45584 entries, 0 to 45583
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45584 non-null  str    
 1   Delivery_person_ID           45584 non-null  str    
 2   Delivery_person_Age          43730 non-null  float64
 3   Delivery_person_Ratings      43676 non-null  float64
 4   Restaurant_latitude          45584 non-null  float64
 5   Restaurant_longitude         45584 non-null  float64
 6   Delivery_location_latitude   45584 non-null  float64
 7   Delivery_location_longitude  45584 non-null  float64
 8   Order_Date                   45584 non-null  str    
 9   Time_Orderd                  43853 non-null  str    
 10  Time_Order_picked            45584 non-null  str    
 11  Weather_conditions           44968 non-null  str    
 12  Road_traffic_density         44983 non-null  str    
 13  Vehicle_condition          

In [4]:
# Cek duplikat baris & ID
print("Baris duplikat:", df.duplicated().sum())
print("ID duplikat:", df['ID'].duplicated().sum())


Baris duplikat: 0
ID duplikat: 0


### 2.2 Model Bisnis: Menelusuri Pola `Delivery_person_ID`

Kolom `Delivery_person_ID` ternyata menyimpan kode kota di depannya, dan mengikuti pola
`[KodeKota]RES[nomor]DEL[nomor]`. Kita cek apakah ID ini konsisten sebagai identitas kurir individu,
atau sekadar kombinasi kode yang di-generate.


In [5]:
# Ekstrak kode kota dari awalan Delivery_person_ID
df["dp_prefix"] = df["Delivery_person_ID"].str.extract(r'^([A-Z]+)')[0]
print("Jumlah kode kota unik:", df["dp_prefix"].nunique())
print(df["dp_prefix"].value_counts())


Jumlah kode kota unik: 22
dp_prefix
JAPRES       3443
RANCHIRES    3228
BANGRES      3193
SURRES       3187
HYDRES       3179
MUMRES       3173
MYSRES       3170
COIMBRES     3169
VADRES       3166
INDORES      3158
CHENRES      3144
PUNERES      3132
AGRRES        763
LUDHRES       758
KNPRES        740
ALHRES        740
DEHRES        737
GOARES        709
AURGRES       703
KOCRES        701
KOLRES        700
BHPRES        691
Name: count, dtype: int64


In [6]:
# Apakah satu Delivery_person_ID konsisten muncul di satu kategori City saja?
dp_city = df.groupby("Delivery_person_ID")["City"].nunique()
print("Kurir yang tercatat di >1 kategori City:", (dp_city > 1).sum(), "dari", len(dp_city))
print("-> Ini tidak masuk akal untuk kurir sungguhan, mengindikasikan ID adalah kombinasi kode sistem,")
print("   bukan identitas individu yang persisten. Delivery_person_ID TIDAK dipakai untuk analisis")
print("   performa kurir individual di notebook ini.")


Kurir yang tercatat di >1 kategori City: 1288 dari 1320
-> Ini tidak masuk akal untuk kurir sungguhan, mengindikasikan ID adalah kombinasi kode sistem,
   bukan identitas individu yang persisten. Delivery_person_ID TIDAK dipakai untuk analisis
   performa kurir individual di notebook ini.


### 2.3 Indikasi Dataset Bersifat Sintetis

In [7]:
# 1) Distribusi Type_of_order yang nyaris presisi rata
print("Distribusi Type_of_order:")
print(df["Type_of_order"].value_counts(normalize=True).round(4) * 100)


Distribusi Type_of_order:
Type_of_order
Snack     25.29
Meal      25.13
Drinks    24.84
Buffet    24.74
Name: proportion, dtype: float64


In [8]:
# 2) Pola waktu antar harian yang berselang-seling ganjil/genap tanggal
tmp = df.copy()
tmp["Order_Date_parsed"] = pd.to_datetime(tmp["Order_Date"], format="%d-%m-%Y")
daily = tmp.groupby("Order_Date_parsed")["Time_taken (min)"].mean().reset_index()
daily["is_even_day"] = daily["Order_Date_parsed"].dt.day % 2 == 0
print(daily.groupby("is_even_day")["Time_taken (min)"].agg(["mean", "std", "count"]).round(2))
print("\n-> Rata-rata waktu antar berbeda signifikan antara tanggal genap vs ganjil dalam bulan,")
print("   pola yang tidak mungkin terjadi secara alami dalam operasional bisnis nyata.")


              mean   std  count
is_even_day                    
False        24.68  2.73     23
True         28.61  2.71     21

-> Rata-rata waktu antar berbeda signifikan antara tanggal genap vs ganjil dalam bulan,
   pola yang tidak mungkin terjadi secara alami dalam operasional bisnis nyata.


In [9]:
# 3) Tidak ada kolom finansial sama sekali
print("Kolom finansial (harga/ongkir/komisi) di dataset:",
      [c for c in df.columns if any(k in c.lower() for k in ['price','fee','cost','amount','revenue'])])


Kolom finansial (harga/ongkir/komisi) di dataset: []


**Kesimpulan Data Understanding:** dataset ini adalah simulasi bisnis *marketplace* tiga sisi
(restoran–pelanggan–kurir) yang kemungkinan besar disintesis untuk keperluan edukasi/kompetisi,
dengan cakupan 20+ kota di India, periode Februari–April 2022, murni operasional (tanpa data finansial).


## 3. Data Quality Assessment

Sebelum membersihkan data, kita petakan dulu semua masalah kualitas data yang ditemukan.


In [10]:
# Missing values per kolom
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Kolom dengan missing values:")
print(missing)
print("\nPersentase:")
print((missing / len(df) * 100).round(2))


Kolom dengan missing values:
Delivery_person_Ratings    1908
Delivery_person_Age        1854
Time_Orderd                1731
City                       1200
multiple_deliveries         993
Weather_conditions          616
Road_traffic_density        601
Festival                    228
dtype: int64

Persentase:
Delivery_person_Ratings    4.19
Delivery_person_Age        4.07
Time_Orderd                3.80
City                       2.63
multiple_deliveries        2.18
Weather_conditions         1.35
Road_traffic_density       1.32
Festival                   0.50
dtype: float64


In [11]:
# Anomali usia kurir
print("Usia kurir < 18 tahun:", (df["Delivery_person_Age"] < 18).sum())
print("Rentang usia:", df["Delivery_person_Age"].min(), "-", df["Delivery_person_Age"].max())


Usia kurir < 18 tahun: 38
Rentang usia: 15.0 - 50.0


In [12]:
# Anomali rating kurir (di luar skala 1-5)
print("Rating > 5.0:", (df["Delivery_person_Ratings"] > 5).sum())
print(sorted(df["Delivery_person_Ratings"].dropna().unique())[-10:])


Rating > 5.0: 53
[np.float64(4.2), np.float64(4.3), np.float64(4.4), np.float64(4.5), np.float64(4.6), np.float64(4.7), np.float64(4.8), np.float64(4.9), np.float64(5.0), np.float64(6.0)]


In [13]:
# Anomali koordinat restoran
print("Koordinat restoran negatif (lat):", (df["Restaurant_latitude"] < 0).sum())
both_zero = (df["Restaurant_latitude"] == 0) & (df["Restaurant_longitude"] == 0)
print("Koordinat restoran (0,0) - tidak valid:", both_zero.sum(),
      f"({both_zero.sum()/len(df)*100:.1f}%)")


Koordinat restoran negatif (lat): 431
Koordinat restoran (0,0) - tidak valid: 3640 (8.0%)


In [14]:
# Format waktu campuran (desimal ala Excel) di Time_Orderd
frac_mask = df["Time_Orderd"].astype(str).str.match(r'^\d?\.\d+$')
print("Baris dengan format desimal Excel (bukan HH:MM):", frac_mask.sum())
print("Contoh nilai:", df.loc[frac_mask, "Time_Orderd"].unique()[:5])


Baris dengan format desimal Excel (bukan HH:MM): 3638
Contoh nilai: <StringArray>
['0.458333333', '0.958333333', '0.791666667', '0.875', '0.375']
Length: 5, dtype: str


## 4. Data Preprocessing

### 4.1 Parsing Tanggal & Waktu

In [15]:
def parse_time_col(s, base=pd.Timestamp("2000-01-01")):
    """Parse kolom waktu yang bercampur format HH:MM dan pecahan hari ala Excel (mis. 0.458333333)."""
    s = s.astype(str)
    is_frac = s.str.match(r'^\d?\.\d+$') | (s == "1")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    normal_mask = ~is_frac & (s != "nan")
    hhmm = pd.to_datetime(s[normal_mask], format="%H:%M", errors="coerce")
    out.loc[normal_mask] = base + pd.to_timedelta(hhmm.dt.hour, unit="h") + pd.to_timedelta(hhmm.dt.minute, unit="m")

    frac_mask = is_frac
    minutes = (s[frac_mask].astype(float) * 24 * 60).round().astype(int) % (24 * 60)
    out.loc[frac_mask] = base + pd.to_timedelta(minutes, unit="m")
    return out

d = df.copy()
d["Order_Date"] = pd.to_datetime(d["Order_Date"], format="%d-%m-%Y")
d["Order_Weekday"] = d["Order_Date"].dt.day_name()
d["Time_Orderd_parsed"] = parse_time_col(d["Time_Orderd"])
d["Time_Order_picked_parsed"] = parse_time_col(d["Time_Order_picked"])
d["Order_Hour"] = d["Time_Orderd_parsed"].dt.hour

print("Time_Orderd berhasil diparse:", d["Time_Orderd_parsed"].notna().sum(), "/", len(d))


Time_Orderd berhasil diparse: 43853 / 45584


### 4.2 Perbaikan Koordinat Restoran

In [16]:
# Nilai negatif adalah bug tanda minus (setelah di-abs, skalanya cocok dengan Delivery_location)
d["Restaurant_latitude"] = d["Restaurant_latitude"].abs()
d["Restaurant_longitude"] = d["Restaurant_longitude"].abs()

# Pasangan (0,0) tidak valid secara geografis -> dijadikan missing
zero_mask = (d["Restaurant_latitude"] == 0) & (d["Restaurant_longitude"] == 0)
d.loc[zero_mask, ["Restaurant_latitude", "Restaurant_longitude"]] = np.nan
print("Koordinat (0,0) dijadikan missing:", zero_mask.sum(), "baris")


Koordinat (0,0) dijadikan missing: 3640 baris


### 4.3 Penanganan Anomali Rating & Usia Kurir

In [17]:
# Rating di luar skala (>5.0) dijadikan missing, akan diimputasi di langkah berikutnya
rating_anomaly = d["Delivery_person_Ratings"] > 5
d.loc[rating_anomaly, "Delivery_person_Ratings"] = np.nan
print("Rating >5 dijadikan missing:", rating_anomaly.sum(), "baris")

# Usia <18 dipertahankan (tidak dihapus), hanya ditandai transparan
d["Age_flag_under18"] = d["Delivery_person_Age"] < 18
print("Usia <18 tahun (dipertahankan + flag):", d["Age_flag_under18"].sum(), "baris")


Rating >5 dijadikan missing:

 53 baris
Usia <18 tahun (dipertahankan + flag): 38 baris


### 4.4 Imputasi Missing Values

In [18]:
# Numerik -> median
for c in ["Delivery_person_Age", "Delivery_person_Ratings"]:
    med = d[c].median()
    n_missing = d[c].isna().sum()
    d[c] = d[c].fillna(med)
    print(f"Imputasi {c}: {n_missing} baris -> median {med}")

# Kategorikal -> label 'Unknown' (transparan, bukan modus, supaya missing tetap terlihat)
for c in ["Weather_conditions", "Road_traffic_density", "City", "Festival"]:
    n_missing = d[c].isna().sum()
    d[c] = d[c].fillna("Unknown")
    print(f"Imputasi {c}: {n_missing} baris -> 'Unknown'")

# multiple_deliveries -> modus
n_missing = d["multiple_deliveries"].isna().sum()
mode_val = d["multiple_deliveries"].mode()[0]
d["multiple_deliveries"] = d["multiple_deliveries"].fillna(mode_val)
print(f"Imputasi multiple_deliveries: {n_missing} baris -> modus {mode_val}")


Imputasi Delivery_person_Age: 1854 baris -> median 30.0
Imputasi Delivery_person_Ratings: 1961 baris -> median 4.7
Imputasi Weather_conditions: 616 baris -> 'Unknown'
Imputasi Road_traffic_density: 601 baris -> 'Unknown'
Imputasi City: 1200 baris -> 'Unknown'
Imputasi Festival: 228 baris -> 'Unknown'
Imputasi multiple_deliveries: 993 baris -> modus 1.0


### 4.5 Feature Engineering

In [19]:
def haversine(lat1, lon1, lat2, lon2):
    """Jarak lurus antara dua titik koordinat (km)."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Distance_km: jarak restoran ke tujuan
d["Distance_km"] = haversine(
    d.Restaurant_latitude, d.Restaurant_longitude,
    d.Delivery_location_latitude, d.Delivery_location_longitude
)

# Prep_Time_min: selisih waktu order dibuat ke waktu kurir pickup ("waktu dapur")
diff = (d["Time_Order_picked_parsed"] - d["Time_Orderd_parsed"]).dt.total_seconds() / 60
diff = np.where(diff < 0, diff + 24 * 60, diff)  # handle wraparound tengah malam
d["Prep_Time_min"] = diff

d = d.drop(columns=["Time_Orderd_parsed", "Time_Order_picked_parsed"])

print("Distance_km:", d["Distance_km"].describe().round(2).to_dict())
print("\nPrep_Time_min:", pd.Series(d["Prep_Time_min"]).describe().round(2).to_dict())


Distance_km: {'count': 41944.0, 'mean': 9.72, 'std': 5.6, 'min': 1.47, '25%': 4.66, '50%': 9.19, '75%': 13.68, 'max': 20.97}

Prep_Time_min: {'count': 43012.0, 'mean': 9.96, 'std': 4.09, 'min': 5.0, '25%': 5.0, '50%': 10.0, '75%': 15.0, 'max': 15.0}


### 4.6 Kluster Anomali: `Festival` yang Hilang Tidak Acak

Sebelum lanjut ke EDA, kita cek baris dengan `Festival = Unknown` (hasil imputasi tahap 4.4).
Ternyata baris ini punya pola waktu antar yang sangat menyimpang.


In [20]:
unk = d[d["Festival"] == "Unknown"]
rest = d[d["Festival"] != "Unknown"]

print("Jumlah baris Festival=Unknown:", len(unk))
print("\nTime_taken pada baris Unknown:")
print(unk["Time_taken (min)"].describe().round(2))
print("\nTime_taken pada baris lainnya:")
print(rest["Time_taken (min)"].describe().round(2))
print("\n-> Waktu antar pada kluster ini sangat seragam (std kecil) dan jauh lebih cepat")
print("   dibanding kelompok manapun -> kemungkinan besar bukan pengiriman normal.")
print("   Baris ini DIKELUARKAN dari analisis utama (bukan diimputasi/diinterpretasi sbg pengiriman biasa).")


Jumlah baris Festival=Unknown: 228

Time_taken pada baris Unknown:
count    228.00
mean      11.17
std        1.23
min       10.00
25%       10.00
50%       11.00
75%       12.00
max       14.00
Name: Time_taken (min), dtype: float64

Time_taken pada baris lainnya:
count    45356.00
mean        26.37
std          9.35
min         10.00
25%         19.00
50%         26.00
75%         32.00
max         54.00
Name: Time_taken (min), dtype: float64

-> Waktu antar pada kluster ini sangat seragam (std kecil) dan jauh lebih cepat
   dibanding kelompok manapun -> kemungkinan besar bukan pengiriman normal.
   Baris ini DIKELUARKAN dari analisis utama (bukan diimputasi/diinterpretasi sbg pengiriman biasa).


In [21]:
d_clean = d.copy()               # dataset hasil cleaning penuh (masih termasuk kluster anomali)
d_eda = d[d["Festival"] != "Unknown"].copy()   # dataset final untuk EDA

print("Dataset hasil cleaning penuh :", d_clean.shape)
print("Dataset final untuk EDA      :", d_eda.shape, f"({len(d) - len(d_eda)} baris kluster anomali dikeluarkan)")


Dataset hasil cleaning penuh : (45584, 26)
Dataset final untuk EDA      : (45356, 26) (228 baris kluster anomali dikeluarkan)


### 4.7 Simpan Dataset Bersih

In [22]:
d_clean.to_csv("zomato_cleaned.csv", index=False)
d_eda.to_csv("zomato_eda.csv", index=False)
print("Tersimpan: zomato_cleaned.csv & zomato_eda.csv")


Tersimpan: zomato_cleaned.csv & zomato_eda.csv


## 5. Exploratory Data Analysis

### 5.1 Distribusi Target Variable (`Time_taken`)

In [23]:
print(d_eda["Time_taken (min)"].describe().round(2))
print("\nSkewness:", d_eda["Time_taken (min)"].skew().round(3))


count    45356.00
mean        26.37
std          9.35
min         10.00
25%         19.00
50%         26.00
75%         32.00
max         54.00
Name: Time_taken (min), dtype: float64

Skewness: 0.492


### 5.2 Ranking Faktor Berdasarkan Pengaruh

In [24]:
# Korelasi faktor numerik terhadap target
num = d_eda.select_dtypes("number").drop(columns=["ID"], errors="ignore")
corr = num.corr()["Time_taken (min)"].round(3).sort_values(key=abs, ascending=False)
print("Korelasi terhadap Time_taken (diurutkan berdasarkan kekuatan):")
print(corr.drop("Time_taken (min)"))


Korelasi terhadap Time_taken (diurutkan berdasarkan kekuatan):
multiple_deliveries            0.376
Delivery_person_Ratings       -0.332
Distance_km                    0.320
Delivery_person_Age            0.290
Vehicle_condition             -0.233
Order_Hour                     0.184
Delivery_location_latitude     0.015
Restaurant_latitude            0.012
Delivery_location_longitude    0.008
Restaurant_longitude           0.006
Prep_Time_min                 -0.004
Name: Time_taken (min), dtype: float64


In [25]:
# Rentang selisih waktu antar untuk faktor kategorikal
factors = {
    "Festival": d_eda,
    "Road_traffic_density": d_eda[d_eda["Road_traffic_density"] != "Unknown"],
    "Weather_conditions": d_eda[d_eda["Weather_conditions"] != "Unknown"],
    "Type_of_vehicle": d_eda,
    "Type_of_order": d_eda,
}
print(f"{'Faktor':<25}{'Rentang (menit)':<18}{'Tercepat':<12}{'Terlambat'}")
for name, subset in factors.items():
    grp = subset.groupby(name)["Time_taken (min)"].mean()
    rng = grp.max() - grp.min()
    print(f"{name:<25}{rng:<18.2f}{grp.idxmin():<12}{grp.idxmax()}")


Faktor                   Rentang (menit)   Tercepat    Terlambat
Festival                 19.53             No          Yes
Road_traffic_density     9.88              Low         Jam
Weather_conditions       6.99              Sunny       Cloudy
Type_of_vehicle          3.12              electric_scootermotorcycle
Type_of_order            0.22              Drinks      Meal


### 5.3 Titik Trade-off: Order Ganda per Kurir (`multiple_deliveries`)

In [26]:
g = d_eda.groupby("multiple_deliveries")["Time_taken (min)"].agg(["count", "mean", "median", "std"]).round(2)
g["kenaikan"] = g["mean"].diff().round(2)
print(g)


                     count   mean  median   std  kenaikan
multiple_deliveries                                      
0.0                  13949  23.00    22.0  8.73       NaN
1.0                  29061  26.76    26.0  8.55      3.76
2.0                   1985  40.45    40.0  4.92     13.69
3.0                    361  47.82    48.0  3.48      7.37


### 5.4 Faktor Eksternal: Traffic & Cuaca

In [27]:
print("Rata-rata waktu antar per kondisi traffic:")
print(d_eda[d_eda["Road_traffic_density"] != "Unknown"]
      .groupby("Road_traffic_density")["Time_taken (min)"].mean().round(2).sort_values())

print("\nRata-rata waktu antar per kondisi cuaca:")
print(d_eda[d_eda["Weather_conditions"] != "Unknown"]
      .groupby("Weather_conditions")["Time_taken (min)"].mean().round(2).sort_values())


Rata-rata waktu antar per kondisi traffic:
Road_traffic_density
Low       21.35
Medium    26.75
High      27.31
Jam       31.23
Name: Time_taken (min), dtype: float64

Rata-rata waktu antar per kondisi cuaca:
Weather_conditions
Sunny         21.97
Sandstorms    25.94
Stormy        25.95
Windy         26.20
Fog           28.94
Cloudy        28.96
Name: Time_taken (min), dtype: float64


### 5.5 Festival: Momen Kritis

In [28]:
festival_avg = d_eda.groupby("Festival")["Time_taken (min)"].agg(["count", "mean"]).round(2)
print(festival_avg)
print(f"\nSelisih: {festival_avg.loc['Yes','mean'] - festival_avg.loc['No','mean']:.2f} menit")

# Cek konsistensi efek festival di semua level traffic
print("\nRata-rata waktu antar: Festival x Traffic")
pivot = d_eda[d_eda["Road_traffic_density"] != "Unknown"].pivot_table(
    index="Road_traffic_density", columns="Festival", values="Time_taken (min)", aggfunc="mean"
).round(2)
print(pivot)


          count   mean
Festival              
No        44460  25.98
Yes         896  45.52

Selisih: 19.54 menit

Rata-rata waktu antar: Festival x Traffic
Festival                 No    Yes
Road_traffic_density              
High                  27.01  45.83
Jam                   30.54  46.09
Low                   21.28  42.02
Medium                26.55  43.72


### 5.6 Faktor Lain: Jarak, Jenis Kendaraan, Jam Sibuk

In [29]:
# Jarak (binned)
d_eda["dist_bin"] = pd.cut(d_eda["Distance_km"], bins=[0, 5, 10, 15, 21])
print("Waktu antar per rentang jarak:")
print(d_eda.groupby("dist_bin", observed=True)["Time_taken (min)"].agg(["count", "mean"]).round(2))


Waktu antar per rentang jarak:
          count   mean
dist_bin              
(0, 5]    11205  22.28
(5, 10]   11237  24.47
(10, 15]  11262  29.89
(15, 21]   8025  29.91


In [30]:
# Jenis kendaraan (catatan: perlu investigasi lanjutan, bisa jadi bias penugasan)
print("Waktu antar per jenis kendaraan:")
print(d_eda.groupby("Type_of_vehicle")["Time_taken (min)"].agg(["count", "mean"]).round(2).sort_values("mean"))


Waktu antar per jenis kendaraan:
                  count   mean
Type_of_vehicle               
electric_scooter   3791  24.55
scooter           15173  24.57
bicycle              68  26.43
motorcycle        26324  27.67


In [31]:
# Pola jam sibuk
print("Waktu antar per jam order:")
print(d_eda.groupby("Order_Hour")["Time_taken (min)"].agg(["count", "mean"]).round(2))


Waktu antar per jam order:
            count   mean
Order_Hour              
0.0           428  22.21
8.0          1797  19.68
9.0          1927  19.63
10.0         1973  19.56
11.0         1948  26.48
12.0          886  26.94
13.0          781  27.57
14.0          791  27.54
15.0          870  23.24
16.0          703  23.05
17.0         4264  27.45
18.0         4463  27.33
19.0         4581  30.87
20.0         4523  31.26
21.0         4675  31.13
22.0         4553  23.26
23.0         4471  22.55


## 6. Insight & Interpretasi

**1. Titik trade-off order ganda** — Waktu antar melonjak +13,69 menit begitu kurir membawa 2 order
tambahan (dari 26,76 ke 40,45 menit), jauh lebih besar dari kenaikan tahap manapun. *Order batching*
efisien sampai 1 order tambahan; di atas itu, penghematan kurir mulai merusak kualitas layanan secara
tidak proporsional. Ini satu-satunya faktor utama yang sepenuhnya berada di bawah kendali perusahaan.

**2. Festival = guncangan kapasitas, bukan sekadar traffic** — Efek festival (+19,54 menit) konsisten
di semua level traffic, mengindikasikan penyebabnya lonjakan volume order (kapasitas), bukan cuma
jalanan lebih macet. Solusinya ada di kesiapan kapasitas kurir, bukan optimasi rute.

**3. Faktor eksternal harus diantisipasi, bukan dihilangkan** — Traffic dan cuaca nyata pengaruhnya
tapi di luar kendali perusahaan. Nilainya ada di prediksi (estimasi ETA dinamis), bukan pencegahan.

**4. Anomali jenis kendaraan** — Motorcycle (27,6 menit) lebih lambat dari scooter/electric_scooter
(~24,5 menit), berlawanan dengan intuisi. Kemungkinan besar bias penugasan (motorcycle ditugaskan untuk
jarak/order lebih kompleks), bukan soal kecepatan kendaraan itu sendiri. Perlu investigasi lanjutan
sebelum jadi dasar kebijakan armada.

**5. Rating & usia kurir: korelasi, bukan kausalitas** — Arah sebab-akibat tidak bisa dipastikan dari
data ini, sehingga tidak boleh jadi dasar kebijakan SDM langsung tanpa studi kausal terpisah.


## 7. Dashboard

Dataset `zomato_eda.csv` yang sudah disiapkan pada bagian 4.7 digunakan sebagai sumber data dashboard
interaktif di Power BI (1 halaman): KPI card, ranking faktor, breakdown `multiple_deliveries`,
`Road_traffic_density`, `Weather_conditions`, dan komposisi `Type_of_order`, dilengkapi slicer
(rentang tanggal, tipe kota, status festival).


## 8. Rekomendasi Insight Bisnis

| # | Rekomendasi | Dampak | Effort | Pemilik |
|---|---|---|---|---|
| 1 | Batasi order ganda maksimal 1 order tambahan per kurir | Tinggi | Rendah | Operasional / Produk |
| 2 | Siapkan kurir cadangan & insentif sebelum festival | Tinggi | Sedang | Operasional / Marketing |
| 3 | Bangun ETA dinamis berbasis traffic & cuaca | Sedang | Sedang–Tinggi | Produk / Data Engineering |
| 4 | Investigasi pola penugasan kendaraan | Sedang | Rendah | Data Analyst |
| 5 | Tahan kebijakan SDM dari rating/usia kurir | Rendah | Rendah | People Analytics / HR |

**Catatan:** karena dataset tidak memiliki data biaya/revenue, dampak di atas bersifat kualitatif,
bukan estimasi finansial.


## 9. Keterbatasan

- Semua temuan bersifat korelasional, bukan sebab-akibat murni.
- `Delivery_person_ID` tidak dipakai untuk klaim performa kurir individual.
- Kesimpulan `City = Semi-Urban` lemah secara statistik (sampel 0,4%).
- 228 baris kluster anomali (`Festival` hilang dengan pola menyimpang) dikeluarkan dari analisis utama.
- Dataset terindikasi bersifat sintetis — insight diposisikan sebagai simulasi kasus bisnis.
- Tidak ada data biaya/revenue — dampak rekomendasi bersifat kualitatif.
